In [1]:
%pip install sentence-transformers chromadb google-generativeai pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb

import google.generativeai as genai

from tqdm import tqdm

D:\python_class\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Anshula\AppData\Local\Temp\ipykernel_44548\3395228278.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
genai.configure(api_key="Enter your API key")
llm = genai.GenerativeModel(
    "models/gemini-2.5-flash"
)

print("Gemini Connected Successfully!")

Gemini Connected Successfully!


In [4]:
DATA_PATH = "Data"

documents = []

for file in os.listdir(DATA_PATH):

    if file.endswith(".txt"):

        file_path = os.path.join(DATA_PATH, file)

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

            documents.append({
                "filename": file,
                "content": text
            })

print("Documents Loaded:", len(documents))

Documents Loaded: 10


In [5]:
for doc in documents:
    print(doc["filename"])

AuraHealth_Dietary_Standards.txt
AuraHealth_Employee_Handbook_2026.txt
BioEnhancement_Ethics_Board_Review.txt
CryoStasis_Recovery_Procedures.txt
Extraterrestrial_Pathogen_Handling.txt
NeuroCrystal_Syndrome_Guidelines.txt
OmniHeal_Memo_And_Project_Details.txt
Quantum_MRI_Operation_Manual.txt
Sector_7_Facility_Security_Protocols.txt
Zyntabulin_Clinical_Trial_Results.txt


**CHUNK DOCUMENTS**

In [6]:
def chunk_text(
    text,
    chunk_size=1000,
    overlap=200
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(
            text[start:end]
        )

        start += chunk_size - overlap

    return chunks

In [7]:
#Create chunks
all_chunks = []

for doc in documents:

    chunks = chunk_text(
        doc["content"]
    )

    for i, chunk in enumerate(chunks):

        all_chunks.append({
            "id": f"{doc['filename']}_{i}",
            "source": doc["filename"],
            "text": chunk
        })

print("Total Chunks:", len(all_chunks))

Total Chunks: 158


In [8]:
#Load embedded model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded!


In [9]:
#Generate embeddings
texts = [
    chunk["text"]
    for chunk in all_chunks
]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings Shape:")
print(embeddings.shape)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings Shape:
(158, 384)


In [10]:
#Create chromaDB
client = chromadb.Client()

collection = client.create_collection(
    name="aurahealth_rag"
)

print("Chroma Collection Created!")

Chroma Collection Created!


In [11]:
#Store chunks
collection.add(
    ids=[
        chunk["id"]
        for chunk in all_chunks
    ],

    documents=[
        chunk["text"]
        for chunk in all_chunks
    ],

    embeddings=embeddings.tolist(),

    metadatas=[
        {
            "source": chunk["source"]
        }
        for chunk in all_chunks
    ]
)

print("Data Stored Successfully!")

Data Stored Successfully!


In [12]:
#Retrieval Function
def retrieve_context(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        query
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

In [13]:
#Test Retrieval
query = "Who is the Head of the OmniHeal initiative?"

results = retrieve_context(query)

for doc in results["documents"][0]:

    print("="*80)
    print(doc)

ants, currently in its seventh iteration, processes terabytes of physiological data in real-time, offering our clinicians unprecedented insights into complex pathologies. These systems are designed to operate symbiotically with human doctors, augmenting their decision-making capabilities rather than replacing them. Strict oversight protocols are in place to ensure that all AI-generated recommendations are reviewed by a senior medical officer before implementation, thereby maintaining the critical human element in healthcare.



### LEADERSHIP AND BUDGET ALLOCATION
The OmniHeal initiative continues to make groundbreaking progress in nanite-assisted surgery, promising to reduce recovery times by up to 80%. We are pleased to announce significant organizational updates regarding the project's leadership and financial backing for the upcoming fiscal year.

- **Head of the OmniHeal Initiative:** Dr. Elena Rostova has officially taken the helm as the Chief Director, bringing her extensive exp

In [14]:
#RAG answer function
def ask_rag(question):

    results = retrieve_context(
    question,
    top_k=5
)

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
You are an assistant for AuraHealth Nexus.

Answer ONLY from the provided context.

If the answer is not present,
say:

"I could not find that information in the documents."

Context:
{context}

Question:
{question}
"""

    response = llm.generate_content(
        prompt
    )

    return response.text

In [15]:
question = """
Who is the Head of the OmniHeal initiative,
and what percentage of the project's
budget is allocated to logistical support?
"""

answer = ask_rag(question)

print(answer)

Dr. Elena Rostova is the Head of the OmniHeal initiative.
18% of the project's budget is allocated to Logistical Support.


In [16]:
response = llm.generate_content(
    "Say Hello"
)

print(response.text)

Hello!


In [18]:
#Interactive chatbot
while True:

    question = input("\nAsk Question: ")

    if question.lower() in [
        "exit",
        "quit"
    ]:
        print("Chat Ended")
        break

    print("\nAnswer:\n")

    print(
        ask_rag(question)
    )

    print("\n" + "="*80)


Ask Question:  exit


Chat Ended



Answer:

I could not find that information in the documents.




Ask Question:  exit


Chat Ended


In [19]:
query = """
Who is the Head of the OmniHeal initiative,
and what percentage of the project's budget
is allocated to logistical support?
"""

results = retrieve_context(query, top_k=5)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n===== CHUNK {i+1} =====\n")
    print(doc)


===== CHUNK 1 =====

ants, currently in its seventh iteration, processes terabytes of physiological data in real-time, offering our clinicians unprecedented insights into complex pathologies. These systems are designed to operate symbiotically with human doctors, augmenting their decision-making capabilities rather than replacing them. Strict oversight protocols are in place to ensure that all AI-generated recommendations are reviewed by a senior medical officer before implementation, thereby maintaining the critical human element in healthcare.



### LEADERSHIP AND BUDGET ALLOCATION
The OmniHeal initiative continues to make groundbreaking progress in nanite-assisted surgery, promising to reduce recovery times by up to 80%. We are pleased to announce significant organizational updates regarding the project's leadership and financial backing for the upcoming fiscal year.

- **Head of the OmniHeal Initiative:** Dr. Elena Rostova has officially taken the helm as the Chief Director, brin

In [20]:
for doc in documents:
    if "OmniHeal" in doc["filename"]:
        print(doc["content"])

INTERNAL MEMO: OMNIHEAL INITIATIVE UPDATE
DATE: October 14, 2025
TO: All Senior Staff and Department Heads
FROM: Board of Directors

The psychological well-being of our staff is just as important as the physical health of our patients. The high-stress environments of our emergency trauma centers and advanced research labs can lead to severe burnout and compassion fatigue. AuraHealth Nexus provides mandatory, confidential counseling services for all employees. Additionally, our facilities feature expansive bio-domes—simulated natural environments designed to offer staff a tranquil space for mental decompression. We recognize that our technological advancements are driven by human ingenuity, which must be protected and nurtured.

The integration of Artificial Intelligence into our daily medical practice has revolutionized patient diagnostics and treatment plans. The MediMind series of AI assistants, currently in its seventh iteration, processes terabytes of physiological data in real-tim

In [21]:
for doc in documents:
    if "OmniHeal" in doc["filename"]:
        print("\nFILE:", doc["filename"])
        print(doc["content"][:5000])  # first 5000 chars


FILE: OmniHeal_Memo_And_Project_Details.txt
INTERNAL MEMO: OMNIHEAL INITIATIVE UPDATE
DATE: October 14, 2025
TO: All Senior Staff and Department Heads
FROM: Board of Directors

The psychological well-being of our staff is just as important as the physical health of our patients. The high-stress environments of our emergency trauma centers and advanced research labs can lead to severe burnout and compassion fatigue. AuraHealth Nexus provides mandatory, confidential counseling services for all employees. Additionally, our facilities feature expansive bio-domes—simulated natural environments designed to offer staff a tranquil space for mental decompression. We recognize that our technological advancements are driven by human ingenuity, which must be protected and nurtured.

The integration of Artificial Intelligence into our daily medical practice has revolutionized patient diagnostics and treatment plans. The MediMind series of AI assistants, currently in its seventh iteration, processe

In [52]:
for doc in documents:
    if "logistical" in doc["content"].lower():
        print("\nFOUND IN:", doc["filename"])
        print(doc["content"])


FOUND IN: AuraHealth_Dietary_Standards.txt
AURAHEALTH INPATIENT DIETARY STANDARDS AND NUTRITION
DEPARTMENT OF PATIENT WELLNESS
FOCUS: POST-OPERATIVE NUTRITION

Our commitment to global health initiatives extends beyond our proprietary facilities. AuraHealth Nexus frequently partners with international aid organizations to deploy rapid-response medical teams to disaster zones. These teams are equipped with portable AI diagnostic units and universal synthetic blood substitutes, allowing them to perform complex triage in the most austere environments. By sharing our technological breakthroughs with those in desperate need, we uphold our core belief that advanced medical care is a universal human right, not a privilege reserved for the few.

Environmental sustainability and facility management are critical components of our long-term strategy. The massive computational power required to run our AI models and the energy-intensive nature of our research laboratories necessitate an innovativ

In [22]:
def retrieve_with_sources(query, top_k=5):

    query_embedding = embedding_model.encode(query)

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results


query = "What is NeuroCrystal Syndrome?"

results = retrieve_with_sources(query)

for i in range(len(results["documents"][0])):

    print("="*80)

    print("Source :", results["metadatas"][0][i]["source"])

    print()

    print(results["documents"][0][i])

Source : NeuroCrystal_Syndrome_Guidelines.txt

e-art facilities across the globe are dedicated to pushing the boundaries of medical science. By fostering a culture of innovation, rigorous ethical standards, and patient-centric care, we have positioned ourselves as the undisputed leader in next-generation healthcare solutions. Our ongoing commitment to research and development ensures that every treatment protocol, every cybernetic enhancement, and every AI diagnostic tool meets the highest possible standards of safety and efficacy.



## PHASE 2 PROGRESSION AND TREATMENT PROTOCOLS

Understanding the pathophysiology of NeuroCrystal Syndrome is critical for our clinical staff. As the syndrome progresses into Phase 2, the crystalline formations begin to interfere significantly with the peripheral nervous system, causing severe neuropathy and localized paralysis. 

### RECOMMENDED TREATMENT FOR PHASE 2
Patients diagnosed with Phase 2 NeuroCrystal Syndrome exhibit moderate crystalline growt

In [23]:
def ask_rag(question):

    results = retrieve_with_sources(
        question,
        top_k=5
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
You are an AI Assistant for AuraHealth Nexus.

Answer ONLY from the given context.

If the answer is not found,
reply exactly:

I could not find that information in the provided documents.

Context:
{context}

Question:
{question}
"""

    response = llm.generate_content(prompt)

    return (
        response.text,
        results["metadatas"][0]
    )

In [25]:
#Test question
question = """
Who is the Head of the OmniHeal Initiative?
"""

answer, sources = ask_rag(question)

print("Answer:\n")

print(answer)

print("\nSources Used:\n")

for src in sources:

    print(src["source"])

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 48.154033174s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 48
}
]

In [56]:
#Interactive chatbot
print("="*70)
print("AuraHealth Nexus RAG Chatbot")
print("Type 'exit' to stop")
print("="*70)

while True:

    question = input("\nAsk Question : ")

    if question.lower() == "exit":
        print("\nGoodbye!")
        break

    answer, sources = ask_rag(question)

    print("\nAnswer:\n")

    print(answer)

    print("\nSources:")

    for src in sources:

        print("-", src["source"])

    print("\n" + "="*70)

AuraHealth Nexus RAG Chatbot
Type 'exit' to stop



Ask Question :  exit



Goodbye!


In [28]:
evaluation_questions = [

    "Who is the Head of the OmniHeal Initiative?",

    "What is NeuroCrystal Syndrome?",

    "What are the symptoms of CryoStasis?",

    "Explain Quantum MRI.",

    "What is Level 3 Sentience?",

    "What are the dietary standards for astronauts?",

    "Explain the employee leave policy.",

    "Describe the security protocol of Sector 7.",

    "What are the clinical trial results of Zyntabulin?",

    "What is the BioEnhancement Ethics Board?"
]

In [29]:
for q in evaluation_questions:

    print("="*100)

    print("Question:")

    print(q)

    print()

    answer, sources = ask_rag(q)

    print("Answer:")

    print(answer)

    print()

    print("Sources:")

    for src in sources:

        print("-", src["source"])

    print()

Question:
Who is the Head of the OmniHeal Initiative?



ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 16.466278974s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 16
}
]

In [30]:
#Save evaluation results
results = []

for q in evaluation_questions:

    answer, sources = ask_rag(q)

    results.append({

        "Question": q,

        "Answer": answer,

        "Sources": ", ".join(
            [
                s["source"]
                for s in sources
            ]
        )
    })

evaluation_df = pd.DataFrame(results)

evaluation_df

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 3.978671517s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 3
}
]

In [31]:
#Save Results to csv
evaluation_df.to_csv(

    "AuraHealth_RAG_Results.csv",

    index=False
)

print("Results saved successfully!")

NameError: name 'evaluation_df' is not defined

In [62]:
#Project summary
print("="*70)

print("PROJECT COMPLETED")

print("="*70)

print(f"Documents Loaded : {len(documents)}")

print(f"Chunks Created : {len(all_chunks)}")

print(f"Embedding Model : all-MiniLM-L6-v2")

print("Vector Database : ChromaDB")

print("LLM : Gemini 2.5 Flash")

print("Status : SUCCESS")

print("="*70)

PROJECT COMPLETED
Documents Loaded : 10
Chunks Created : 158
Embedding Model : all-MiniLM-L6-v2
Vector Database : ChromaDB
LLM : Gemini 2.5 Flash
Status : SUCCESS


**History save**

In [ ]:
chat_history = []

def ask_rag_memory(question):

    results = retrieve_with_sources(
        question,
        top_k=5
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    history = "\n".join(
        chat_history[-5:]
    )

    prompt = f"""
Previous Conversation:
{history}

Context:
{context}

Question:
{question}

Answer only from the context.
"""

    response = llm.generate_content(prompt)

    answer = response.text

    chat_history.append(
        f"User: {question}"
    )

    chat_history.append(
        f"Assistant: {answer}"
    )

    return answer